# Validation on MARS (automatically tracked, same clothes)

**Why MARS:** its tracklets come from an automatic detector and tracker (Zheng et al., ECCV 2016), like MEVID's. Its test set even contains 870 "junk" tracklets where the box missed the person. iLIDS-VID and PRID-2011 were cropped by hand, so they barely contain this problem.

**Data:**
- **Images:** download **bbox_test** from the [official MARS page](https://zheng-lab-anu.github.io/Project/project_mars.html) (ANU OneDrive link) and upload the zip to `Data/MARS/` on Drive.
- **Annotations:** the official MARS-evaluation files from GitHub, fetched automatically.

**Same method and settings as before; all three models zero-shot.**
- Variants: all frames, **ours**, random removal, and feature-distance removal. The two controls remove as many frames as ours.
- **Retrieval:** the official MARS protocol (mAP, Rank-1, Rank-5).
- **Consistent ID:** computed on the 634 real identities, over 20 tests. MARS is six times larger than MEVID-SC, so 10 splits × 2 directions instead of 25 × 2.

**Run order on an A100 (L4 works, but slower):**

| Cell | What | Time |
|---|---|---|
| 1 | Unzip | about 10 min |
| 2 | Pose check | about 20 min |
| 3 | Our keep rule | seconds |
| 4 | FastReID and OSNet | about 25 min |
| 5–6 | CSCI: all frames, ours, random, feature | about 1 to 1.5 hours each |
| 7 | **Standalone results.** Shows whatever has finished | |

Everything is saved to `MEVID-results/Validation-MARS` on Drive. Reruns skip finished parts.

**Cite:** Zheng et al., *MARS: A Video Benchmark for Large-Scale Person Re-Identification*, ECCV 2016.

In [ ]:
# 1. Setup: MARS test set (auto-detected and tracked; Zheng et al., ECCV
#    2016). Images: Data/MARS/ on Drive (bbox_test zip or folder, from the
#    official project page). Annotations: the official MARS-evaluation files
#    (GitHub), fetched automatically. Same method and settings as before.
import os, glob, json, time, shutil, subprocess, sys, zipfile, warnings
import numpy as np, pandas as pd, scipy.io as sio
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(pats[0])

CSCI_DIR = os.path.dirname(find([f'{SD}/Data/MEVID-results/CSCI/ez_eva02_vid_hybrid_extra_best.pth',
                                 f'{SD}/**/CSCI/ez_eva02_vid_hybrid_extra_best.pth']))
WT = f'{CSCI_DIR}/ez_eva02_vid_hybrid_extra_best.pth'
RES = os.path.dirname(CSCI_DIR); DATA = os.path.dirname(RES)
OUT = f'{RES}/Validation-MARS'; os.makedirs(OUT, exist_ok=True)
SETTINGS = {'dataset': 'MARS test split: 12,180 tracklets, 634 people, 6 cameras, '
                       'including 870 junk tracklets (Zheng et al., ECCV 2016)',
            'models': 'CSCI (MEVID weights), FastReID SBS R101-IBN (MSMT17), '
                      'OSNet x1_0 (MSMT17); all zero-shot, unchanged',
            'check_every': 4, 'pose_model': 'yolo11m-pose.pt', 'imgsz': 320,
            'det_conf': 0.25, 'keypoint_conf': 0.5, 'visible_thr': 0.5,
            'min_kept_frames': 8, 'precision_target': 0.90,
            'n_splits': 10,   # 20 tests: MARS is 6x larger than MEVID-SC
            'seed': 2027, 'control_seed': 2027,
            'calibrator': 'largest cut with pair precision >= target, '
                          'geomspace(0.001, 1.5, 120)',
            'retrieval': 'official MARS protocol: junk = id -1 and same id & '
                         'same camera; distractors (id 0) count as wrong',
            'consistent_id': 'tracklets of the 634 real identities only'}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
st = SETTINGS

# official annotations
GH = 'https://raw.githubusercontent.com/liangzheng06/MARS-evaluation/master/info'
for fn in ['tracks_test_info.mat', 'query_IDX.mat', 'test_name.txt']:
    if not os.path.exists(f'{OUT}/{fn}'):
        subprocess.run(['wget', '-q', '-O', f'{OUT}/{fn}', f'{GH}/{fn}'], check=True)
T = sio.loadmat(f'{OUT}/tracks_test_info.mat')['track_test_info'].astype(np.int64)
Q = sio.loadmat(f'{OUT}/query_IDX.mat')['query_IDX'].ravel().astype(np.int64) - 1
names = [l.strip() for l in open(f'{OUT}/test_name.txt') if l.strip()]
info = np.stack([T[:, 0] - 1, T[:, 1] - 1, T[:, 2], np.ones(len(T), np.int64), T[:, 3]], 1)
qidx = Q; gidx = np.setdiff1d(np.arange(len(info)), qidx)
pid, cam = info[:, 2], info[:, 4]

# images: local copy of bbox_test (unzip once per session)
LOCAL = '/content/mars'
BBOX = sorted(glob.glob(f'{LOCAL}/**/bbox_test', recursive=True))
if not BBOX:
    src = sorted(glob.glob(f'{DATA}/MARS/*bbox_test*.zip') + glob.glob(f'{DATA}/MARS/*.zip'))
    if src:
        t0 = time.time(); print('Unzipping', src[0], '(about 10 min) ...')
        os.makedirs(LOCAL, exist_ok=True)
        subprocess.run(['unzip', '-q', '-o', src[0], '-d', LOCAL], check=True)
        print(f'done in {time.time() - t0:.0f} s')
    elif glob.glob(f'{DATA}/MARS/**/bbox_test', recursive=True):
        print('Copying the bbox_test folder from Drive (slow) ...')
        shutil.copytree(glob.glob(f'{DATA}/MARS/**/bbox_test', recursive=True)[0],
                        f'{LOCAL}/bbox_test')
    else:
        raise FileNotFoundError(f'Put the MARS bbox_test zip in {DATA}/MARS/ '
                                '(from https://zheng-lab-anu.github.io/Project/project_mars.html)')
    BBOX = sorted(glob.glob(f'{LOCAL}/**/bbox_test', recursive=True))
BBOX = BBOX[0]
path = lambda k: f'{BBOX}/{names[k][:4]}/{names[k]}'
assert os.path.exists(path(0)) and os.path.exists(path(len(names) - 1)), 'image layout'

# MEVID layout for CSCI (names already follow <id4>/<name>)
base = '/content/mars_root_all'; os.makedirs(base, exist_ok=True)
open(f'{base}/test_name.txt', 'w').write('\n'.join(names) + '\n')
np.savetxt(f'{base}/track_test_info.txt', info, fmt='%d')
np.savetxt(f'{base}/query_IDX.txt', qidx, fmt='%d')
np.savez(f'{OUT}/MARS_meta.npz', info=info, qidx=qidx, gidx=gidx)
print(f'MARS test: {len(info):,} tracklets, {len(np.unique(pid[pid > 0]))} people, '
      f'{len(qidx)} queries, {len(names):,} frames, junk {int((pid == -1).sum())}, '
      f'distractors {int((pid == 0).sum())} | results -> {OUT}')


In [ ]:
# 2. Pose check on every 4th frame (resumable; saved to Drive in chunks).
#    About 170,000 frames: 15 to 25 min.
check = []
for s, e in info[:, :2]:
    c = list(range(s, e + 1, st['check_every']))
    if c[-1] != e:
        c.append(e)
    check += c
K = np.unique(np.array(check))
done = {}
for f in sorted(glob.glob(f'{OUT}/personcheck_full_*.npz')):
    p = np.load(f)
    for n, fo, vi in zip(p['name'], p['found'], p['vis']):
        done[str(n)] = (int(fo), float(vi))
todo = [k for k in K if names[k] not in done]
print(f'{len(K):,} frames to check; {len(K) - len(todo):,} reused; '
      f'{len(todo):,} new')
if todo:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'],
                   check=True)
    import cv2, torch
    from ultralytics import YOLO
    from torch.utils.data import Dataset, DataLoader
    from tqdm.auto import tqdm

    class Imgs(Dataset):
        def __init__(self, ks): self.ks = ks
        def __len__(self): return len(self.ks)
        def __getitem__(self, i): return cv2.imread(path(self.ks[i]))

    pose = YOLO(st['pose_model'])
    CH = 20000
    start = len(glob.glob(f'{OUT}/personcheck_full_*.npz'))
    for j, c0 in enumerate(range(0, len(todo), CH)):
        ks = todo[c0:c0 + CH]
        fo = np.zeros(len(ks), np.int8); vi = np.zeros(len(ks), np.float32); i = 0
        for batch in tqdm(DataLoader(Imgs(ks), batch_size=128, num_workers=8,
                                     collate_fn=lambda b: b), leave=False,
                          desc=f'chunk {j + 1}/{-(-len(todo) // CH)}'):
            for im, r in zip(batch, pose.predict(batch, imgsz=st['imgsz'],
                                                 conf=st['det_conf'], verbose=False)):
                if len(r.boxes):
                    H, W = im.shape[:2]
                    xy = r.boxes.xyxy.cpu().numpy(); cf = r.boxes.conf.cpu().numpy()
                    area = (xy[:, 2] - xy[:, 0]) * (xy[:, 3] - xy[:, 1]) / (H * W)
                    m = int(np.argmax(area * cf))
                    fo[i] = 1
                    vi[i] = (r.keypoints.data.cpu().numpy()[m][:, 2]
                             >= st['keypoint_conf']).mean()
                i += 1
        np.savez(f'{OUT}/personcheck_full_{start + j:03d}.npz',
                 name=np.array([names[k] for k in ks]), found=fo, vis=vi)
        for k, a, b in zip(ks, fo, vi):
            done[names[k]] = (int(a), float(b))
FOUND = np.array([done[names[k]][0] for k in K], np.int8)
VIS = np.array([done[names[k]][1] for k in K], np.float32)
print(f'person found in {100 * FOUND.mean():.1f}% of checked frames')


In [ ]:
# 3. OUR keep-mask (same rule as MEVID-SC), on all frames and on the
#    checked frames, and the number of frames each control removes.
RANGES, KEEP_C, KEEP_ALL = [], np.ones(len(K), bool), []
for t, (s, e, *_) in enumerate(info):
    lo, hi = np.searchsorted(K, s), np.searchsorted(K, e, 'right')
    RANGES.append((lo, hi))
    fr = np.arange(s, e + 1)
    near = lo + np.abs(fr[:, None] - K[lo:hi][None, :]).argmin(1)
    keep = (FOUND[near] == 1) & (VIS[near] >= st['visible_thr'])
    if keep.sum() < st['min_kept_frames']:
        keep[:] = True                                  # clip unchanged
    else:
        KEEP_C[lo:hi] = (FOUND[lo:hi] == 1) & (VIS[lo:hi] >= st['visible_thr'])
        if not KEEP_C[lo:hi].any():
            KEEP_C[lo:hi] = True
    KEEP_ALL.append(keep)
REMOVE_ALL = np.array([(~k).sum() for k in KEEP_ALL])   # CSCI (all frames)
REMOVE_C = np.array([(~KEEP_C[lo:hi]).sum() for lo, hi in RANGES])
print(f'ours removes {REMOVE_ALL.sum():,} of {len(names):,} frames '
      f'({100 * REMOVE_ALL.sum() / len(names):.1f}%) in '
      f'{(REMOVE_ALL > 0).sum()} of {len(info)} clips')
np.savez(f'{OUT}/keep_summary.npz', remove_all=REMOVE_ALL, remove_checked=REMOVE_C)


In [ ]:
# 4. Image models (FastReID SBS R101-IBN and OSNet, both MSMT17): frame
#    features on the checked frames, then 4 fingerprints per clip (all /
#    ours / random / feature-distance). OSNet frame features are also kept
#    on Drive for CSCI's feature-distance control. About 30 min on an A100.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchreid',
                'gdown', 'yacs', 'termcolor', 'tabulate'], check=True)
import torch, torch.nn.functional as F
import torchvision.transforms as TV
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)


class Frames(Dataset):
    def __init__(self, tf): self.tf = tf
    def __len__(self): return len(K)
    def __getitem__(self, i): return self.tf(Image.open(path(K[i])).convert('RGB'))


def run(model, tf, dim, fpath, bs):
    if os.path.exists(fpath):
        return np.load(fpath, mmap_mode='r')
    out = np.lib.format.open_memmap(fpath + '.tmp.npy', 'w+', np.float16, (len(K), dim))
    i = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for x in tqdm(DataLoader(Frames(tf), batch_size=bs, num_workers=8,
                                 pin_memory=True), desc=os.path.basename(fpath)):
            f = model(x.cuda(non_blocking=True))
            f = F.normalize((f[0] if isinstance(f, (tuple, list)) else f).float(), dim=1)
            out[i:i + len(f)] = f.cpu().numpy(); i += len(f)
    out.flush(); del out
    os.rename(fpath + '.tmp.npy', fpath)
    return np.load(fpath, mmap_mode='r')


VARIANTS = ['all frames', 'ours (pose: person visible)',
            'control: random frames', 'control: feature-distance']


def fingerprints(Fm, short):
    fn = f'{OUT}/fingerprints_{short}.npz'
    if os.path.exists(fn):
        return
    Fm = np.asarray(Fm, dtype=np.float32)
    rng = np.random.default_rng(st['control_seed'])
    out = {v: np.zeros((len(info), Fm.shape[1]), np.float32) for v in VARIANTS}
    for t, (lo, hi) in enumerate(RANGES):
        f = Fm[lo:hi]; k = REMOVE_C[t]
        out[VARIANTS[0]][t] = f.mean(0)
        out[VARIANTS[1]][t] = f[KEEP_C[lo:hi]].mean(0)
        r = np.ones(hi - lo, bool)
        if k:
            r[rng.choice(hi - lo, k, replace=False)] = False
        out[VARIANTS[2]][t] = f[r].mean(0)
        mu = unit(f.mean(0, keepdims=True))[0]
        d = np.ones(hi - lo, bool)
        if k:
            d[np.argsort(-(1 - f @ mu), kind='stable')[:k]] = False
        out[VARIANTS[3]][t] = f[d].mean(0)
    np.savez(fn, **out, pid=pid, cam=cam)
    print('saved', fn)


# OSNet x1_0 (MSMT17): frame features saved on Drive (~0.3 GB)
import shutil
OSF = f'{OUT}/osnet_checked_frames.npy'          # Drive copy (for restarts)
OSL = '/content/osnet_checked_frames.npy'          # written locally first
if os.path.exists(OSF) and not os.path.exists(OSL):
    shutil.copy(OSF, OSL)
if not (os.path.exists(OSL) and os.path.exists(f'{OUT}/fingerprints_OSNet.npz')):
    import torchreid, gdown
    m = torchreid.models.build_model(name='osnet_x1_0', num_classes=1000,
                                     loss='softmax', pretrained=True)
    ck = os.path.expanduser('~/.cache/torch/checkpoints/osnet_x1_0_msmt17.pth')
    os.makedirs(os.path.dirname(ck), exist_ok=True)
    for url in ['https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M',
                'https://drive.google.com/uc?id=1IosIFlLiulGIjwW3H8uMRmx3MzPwf86x']:
        if not os.path.exists(ck):
            try:
                gdown.download(url, ck, quiet=True, fuzzy=True)
            except Exception as ex:
                print('download failed:', ex)
    torchreid.utils.load_pretrained_weights(m, ck)
    tf = TV.Compose([TV.Resize((256, 128)), TV.ToTensor(),
                     TV.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    fingerprints(run(m.cuda().eval(), tf, 512, OSL, 512), 'OSNet')
    del m; torch.cuda.empty_cache()
if not os.path.exists(OSF):
    shutil.copy(OSL, OSF)

# FastReID SBS R101-IBN (MSMT17): frame features local only (large)
if not os.path.exists(f'{OUT}/fingerprints_FastReID.npz'):
    FRP = '/content/fast-reid'
    if not os.path.isdir(FRP):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/JDAI-CV/fast-reid.git',
                        FRP], check=True)
        for f in [f'{FRP}/fastreid/data/build.py', f'{FRP}/fastreid/evaluation/testing.py']:
            s = open(f).read()
            s = s.replace('from collections import Mapping, OrderedDict',
                          'from collections import OrderedDict\nfrom collections.abc import Mapping')
            s = s.replace('from collections import Mapping', 'from collections.abc import Mapping')
            open(f, 'w').write(s)
    WFR = '/content/msmt_sbs_R101-ibn.pth'
    if not os.path.exists(WFR):
        subprocess.run(['wget', '-q', '-O', WFR, 'https://github.com/JDAI-CV/fast-reid/'
                        'releases/download/v0.1.1/msmt_sbs_R101-ibn.pth'], check=True)
    sys.path.insert(0, FRP)
    from fastreid.config import get_cfg
    from fastreid.modeling.meta_arch import build_model
    from fastreid.utils.checkpoint import Checkpointer
    cfg = get_cfg(); cfg.merge_from_file(f'{FRP}/configs/MSMT17/sbs_R101-ibn.yml')
    cfg.MODEL.BACKBONE.PRETRAIN = False; cfg.freeze()
    m = build_model(cfg).cuda().eval(); Checkpointer(m).load(WFR)
    tf = TV.Compose([TV.Resize((384, 128)), TV.PILToTensor(),
                     TV.Lambda(lambda t: t.float())])
    fingerprints(run(m, tf, 2048, '/content/frames_fastreid.npy', 256), 'FastReID')
    del m; torch.cuda.empty_cache()
print('image-model fingerprints ready')


In [ ]:
# 5. CSCI: authors' code and released MEVID weights, unchanged (same
#    patches as before), and three filtered datasets: ours, random,
#    feature-distance (same number of frames removed per clip as ours).
REPO = '/content/ICCV-CSCI-Person-ReID'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ppriyank/'
                    'ICCV-CSCI-Person-ReID.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.9',
                'einops', 'yacs', 'mmengine', 'decord', 'tensorboardX',
                'tensorboard', 'termcolor', 'tabulate', 'h5py', 'scikit-learn',
                'lmdb', 'albumentations', 'torchinfo', 'grad-cam',
                'opencv-python-headless'], check=True)
f = f'{REPO}/processor/eval_mevid.py'; src = open(f).read()
anchor = '    distmat = distmat.cpu()\n'
if 'CSCI_DUMP' not in src:
    assert src.count(anchor) == 1, 'eval_mevid.py changed upstream'
    src = src.replace(anchor, anchor +
        "    if local_rank == 0 and os.environ.get('CSCI_DUMP'):\n"
        "        import numpy as _np\n"
        "        _np.savez(os.environ['CSCI_DUMP'], qf=qf.float().cpu().numpy(), "
        "gf=gf.float().cpu().numpy(), q_pids=q_pids.numpy(), g_pids=g_pids.numpy(), "
        "q_camids=q_camids.numpy(), g_camids=g_camids.numpy())\n")
    if '\nimport os' not in src:
        src = 'import os\n' + src
    open(f, 'w').write(src)
f = f'{REPO}/data/dataset_loader.py'; src = open(f).read()
old = '        self.return_index = return_index\n        self.__getitem__(0)\n'
if old in src:
    open(f, 'w').write(src.replace(old, '        self.return_index = return_index\n'
                                        '        if not train:\n            self.__getitem__(0)\n'))

OSN = np.load('/content/osnet_checked_frames.npy' if os.path.exists(
    '/content/osnet_checked_frames.npy') else f'{OUT}/osnet_checked_frames.npy', mmap_mode='r')
rng = np.random.default_rng(st['control_seed'])
for v in ['visible', 'random', 'feature']:
    root = f'/content/mars_root_{v}'
    if os.path.exists(f'{root}/track_test_info.txt'):
        continue
    new_names, new_info = [], []
    for t, (s, e, p, o, c) in enumerate(info):
        fr = np.arange(s, e + 1); k = REMOVE_ALL[t]
        if v == 'visible':
            keep = KEEP_ALL[t]
        else:
            keep = np.ones(len(fr), bool)
            if k and v == 'random':
                keep[rng.choice(len(fr), k, replace=False)] = False
            elif k:
                lo, hi = RANGES[t]
                near = lo + np.abs(fr[:, None] - K[lo:hi][None, :]).argmin(1)
                E = np.asarray(OSN[lo:hi], np.float32)
                mu = E.mean(0); mu /= np.linalg.norm(mu)
                dist = 1 - np.asarray(OSN[near], np.float32) @ mu
                keep[np.argsort(-dist, kind='stable')[:k]] = False
        assert (~keep).sum() == k
        a = len(new_names)
        new_names += [names[j] for j in fr[keep]]
        new_info.append([a, len(new_names) - 1, p, o, c])
    os.makedirs(root, exist_ok=True)
    open(f'{root}/test_name.txt', 'w').write('\n'.join(new_names) + '\n')
    np.savetxt(f'{root}/track_test_info.txt', np.array(new_info), fmt='%d')
    np.savetxt(f'{root}/query_IDX.txt', qidx, fmt='%d')
    TRN = sorted(glob.glob(f'{DATA}/MEVID-SameClothes/annotations/track_train_info.txt'))[0]
    for n, src_ in {'train_name.txt': TRN.replace('track_train_info', 'train_name'),
                    'track_train_info.txt': TRN, 'bbox_test': BBOX}.items():
        if not os.path.exists(f'{root}/{n}'):
            os.symlink(src_, f'{root}/{n}')
    print(f'{v}: {len(new_names):,} frames kept')
TRN = sorted(glob.glob(f'{DATA}/MEVID-SameClothes/annotations/track_train_info.txt'))[0]
for n, src_ in {'train_name.txt': TRN.replace('track_train_info', 'train_name'),
                'track_train_info.txt': TRN, 'bbox_test': BBOX}.items():
    if not os.path.lexists(f'{base}/{n}'):
        os.symlink(src_, f'{base}/{n}')


In [ ]:
# 6. Run CSCI: all frames (baseline), ours, and the two controls, with a live progress bar.
#    Saved to Drive when each finishes; a rerun skips finished ones.
from tqdm.auto import tqdm
f = f'{REPO}/processor/eval_mevid.py'; src = open(f).read()
for loop in ['    for n_iter, (imgs, pids, camids, clothes_id, clothes_ids, meta) in enumerate(dataloader):\n',
             '    for n_iter, (imgs, pids, camids, clothes_id, clothes_ids, meta, index) in enumerate(dataloader):\n']:
    tag = loop + "        if n_iter % 10 == 0 or n_iter == len(dataloader) - 1: print(f'CSCI_PROGRESS {n_iter + 1} {len(dataloader)}', flush=True)\n"
    if loop in src and 'CSCI_PROGRESS' not in src.split(loop)[1][:200]:
        src = src.replace(loop, tag)
open(f, 'w').write(src)

for v in ['all', 'visible', 'random', 'feature']:
    OUTF = f'{OUT}/test_templates_csci_{v}.npz'
    if os.path.exists(OUTF):
        print('done already:', v); continue
    DUMP = f'{OUT}/csci_dump_{v}.npz'
    if not os.path.exists(DUMP):
        env = dict(os.environ, CSCI_DUMP=DUMP, PYTHONPATH=REPO, PYTHONUNBUFFERED='1')
        cmd = ['torchrun', '--nproc_per_node=1', '--master_port=12391',
               'train_two_step.py', '--eval', '--resume',
               '--config_file', 'configs/mevid_eva02_l_cloth.yml',
               'DATA.ROOT', (f'/content/mars_root_{v}/' if v != 'all' else f'{base}/'), 'TEST.WEIGHT', WT,
               'SOLVER.SEED', '12345', 'TRAIN.TRAIN_VIDEO', 'True',
               'DATA.DATASET', 'mevid', 'MODEL.NAME', 'ez_eva02_vid_hybrid_extra',
               'TRAIN.COLOR_PROFILE', '17', 'DATA.TEST_BATCH', '16',
               'DATA.NUM_WORKERS', '2']
        t0 = time.time(); bar = None; phase = 0
        names_ = ['query clips', 'gallery clips']
        print(f'Starting CSCI ({v}); loading data takes a few minutes before the bar appears ...')
        with open(f'{OUT}/csci_log_{v}.txt', 'w') as log:
            p = subprocess.Popen(cmd, cwd=REPO, env=env, text=True,
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            for line in p.stdout:
                log.write(line)
                if line.startswith('CSCI_PROGRESS'):
                    done_, total = map(int, line.split()[1:3])
                    if bar is None or total != bar.total or done_ < bar.n:
                        if bar is not None:
                            bar.close()
                        bar = tqdm(total=total, desc=f'{v}: {names_[min(phase, 1)]}')
                        phase += 1
                    bar.update(done_ - bar.n)
                elif any(k in line for k in ['top1', 'mAP', 'Error', 'Traceback',
                                             'out of memory', 'Killed']):
                    print(line.rstrip())
            p.wait()
            if bar is not None:
                bar.close()
        print(f'{v}: {(time.time() - t0) / 60:.0f} min | CSCI exit code {p.returncode} '
              '(0 = fine; -9 = killed for memory; non-zero after the dump is harmless)')
    assert os.path.exists(DUMP), (f'CSCI stopped before saving features ({v}). '
                                  f'See {OUT}/csci_log_{v}.txt')
    z = np.load(DUMP)
    assert (z['q_pids'] == pid[qidx]).all() and (z['g_pids'] == pid[gidx]).all()
    T = np.zeros((len(info), z['qf'].shape[1]), np.float32)
    T[qidx], T[gidx] = z['qf'], z['gf']
    np.savez(OUTF, templates=T, pid=pid, cam=cam)
    print('saved', OUTF)


In [ ]:
# 7. STANDALONE results for MARS (works after a restart; needs only Drive).
import os, glob, zipfile, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
from tqdm.auto import tqdm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon
drive.mount('/content/drive')
OUT = sorted(glob.glob('/content/drive/Shareddrives/*/Data/MEVID-results/Validation-MARS'))[0]
meta = np.load(f'{OUT}/MARS_meta.npz')
info, qidx = meta['info'], meta['qidx']
pid, cam = info[:, 2], info[:, 4]
st = {'precision_target': 0.90, 'n_splits': 10, 'seed': 2027}
def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)
V = ['all frames', 'ours (pose: person visible)', 'control: random frames',
     'control: feature-distance']
MODELS = ['CSCI (MEVID weights)', 'FastReID SBS R101-IBN (MSMT17)', 'OSNet x1_0 (MSMT17)']
FEATS = {}
for v, fn in zip(V, ['all', 'visible', 'random', 'feature']):
    p = f'{OUT}/test_templates_csci_{fn}.npz'
    if os.path.exists(p):
        FEATS[(MODELS[0], v)] = unit(np.load(p)['templates'])
    else:
        print('not run yet: CSCI', v)
for mname, short in zip(MODELS[1:], ['FastReID', 'OSNet']):
    p = f'{OUT}/fingerprints_{short}.npz'
    if os.path.exists(p):
        z = np.load(p)
        for v in V:
            FEATS[(mname, v)] = unit(z[v])

def retrieval(X):
    """official MARS protocol: gallery = all other tracklets; junk = id -1
    and same id in the same camera; distractors (id 0) count as wrong"""
    aps, r1, r5 = [], [], []
    for q in qidx:
        s = X @ X[q]
        keep = (np.arange(len(pid)) != q) & (pid != -1) & ~((pid == pid[q]) & (cam == cam[q]))
        idx = np.flatnonzero(keep)
        order = idx[np.argsort(-s[idx])]
        hits = (pid[order] == pid[q])
        if not hits.any():
            continue
        r = np.flatnonzero(hits)
        aps.append(np.mean((np.arange(len(r)) + 1) / (r + 1)))
        r1.append(float(r[0] == 0)); r5.append(float(r[0] < 5))
    return {'mAP %': 100 * np.mean(aps), 'Rank-1 %': 100 * np.mean(r1),
            'Rank-5 %': 100 * np.mean(r5)}

ID = np.flatnonzero(pid > 0)              # consistent ID: real identities only
def pair_precision(lab, p):
    _, pi = np.unique(p, return_inverse=True); _, ci = np.unique(lab, return_inverse=True)
    cell = np.bincount(pi * (ci.max() + 1) + ci).astype(float)
    nc = np.bincount(ci).astype(float)
    same = (nc * (nc - 1)).sum()
    return 100 * (cell * (cell - 1)).sum() / same if same else 100.0

def score(pred, p, c):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1)); np.add.at(M, (pi, ci), 1)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0); tp, same = (M * (M - 1)).sum(), (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    Kc = np.zeros((M.shape[0], ki.max() + 1, M.shape[1])); np.add.at(Kc, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (Kc.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (Kc ** 2).sum((1, 2))).sum()
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'pair precision %': 100 * tp / same if same else 100.0,
            'cross-camera recall %': 100 * x_same / max(1, x_tot)}

def tree(X, idx):
    D = np.clip(1 - X[idx] @ X[idx].T, 0, None); D = (D + D.T) / 2
    np.fill_diagonal(D, 0)
    return linkage(squareform(D, checks=False), 'average')

CUTS = np.geomspace(0.001, 1.5, 120)
def calibrate(L, idx):
    best = CUTS[0]
    for t in CUTS:
        if pair_precision(fcluster(L, t, 'distance'), pid[idx]) >= 100 * st['precision_target']:
            best = t
    return best

people = np.unique(pid[ID])
rng = np.random.default_rng(st['seed'])
SPL = []
for s in range(st['n_splits']):
    perm = rng.permutation(people)
    a = np.isin(pid[ID], perm[:len(people) // 2])
    SPL.append([ID[a], ID[~a]])
ROWS = []
for (mname, v), X in tqdm(FEATS.items(), desc='models x variants'):
    acc = retrieval(X)
    for s, G in enumerate(SPL):
        Ls = [tree(X, g) for g in G]
        for c, t in [(0, 1), (1, 0)]:
            r = score(fcluster(Ls[t], calibrate(Ls[c], G[c]), 'distance'),
                      pid[G[t]], cam[G[t]])
            r.update(model=mname, variant=v, split=s, direction=c, **acc)
            ROWS.append(r)
REP = pd.DataFrame(ROWS)
REP.to_csv(f'{OUT}/mars_all_tests.csv', index=False)
COLS = ['IDF1 %', 'IDs per person', 'cross-camera recall %', 'pair precision %',
        'mAP %', 'Rank-1 %', 'Rank-5 %']
MAIN = REP.groupby(['model', 'variant'], sort=False)[COLS].mean().round(2)
print(f'MARS, mean over {2 * st["n_splits"]} tests (retrieval: official protocol):')
display(MAIN)
MAIN.to_csv(f'{OUT}/mars_summary.csv')
PAIR = []
for mname in MODELS:
    R = REP[REP.model == mname]
    if (R.variant == V[1]).sum() == 0:
        continue
    o = R[R.variant == V[1]].set_index(['split', 'direction'])
    for ref in [V[0], V[2], V[3]]:
        if (R.variant == ref).sum() == 0:
            continue
        b = R[R.variant == ref].set_index(['split', 'direction'])
        for metric, better in [('IDF1 %', 1), ('IDs per person', -1),
                               ('cross-camera recall %', 1)]:
            d = o[metric] - b[metric]
            PAIR.append({'model': mname.split(' (')[0], 'ours vs': ref, 'metric': metric,
                         'mean change': round(d.mean(), 2),
                         'ours better in % of tests': round(100 * (better * d > 0).mean()),
                         'p (Wilcoxon)': f'{wilcoxon(d).pvalue:.1e}' if (d != 0).any() else 'n/a'})
PAIR = pd.DataFrame(PAIR); PAIR.to_csv(f'{OUT}/mars_paired.csv', index=False)
print('\nOurs vs all frames and the controls:'); display(PAIR)
with zipfile.ZipFile('/content/MARS_results.zip', 'w') as zf:
    for f in glob.glob(f'{OUT}/mars_*.csv') + glob.glob(f'{OUT}/*.json'):
        zf.write(f, os.path.basename(f))
print('Saved to', OUT, '\nZip: /content/MARS_results.zip')
